# Joins — Skew Distribution Report

Per-key **count**, **% of total**, **skew factor** (count ÷ average), and **is_skewed** flag on `silver.order_items`. Top 10 keys for `seller_id` and `product_id`.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.joins.skew_report as skew_report_module
importlib.reload(skew_report_module)

from src.joins.business_questions import SilverJoinTables
from src.joins.skew_report import run_skew_distribution_report
from src.spark_performance.skew import analyze_key_skew, DEFAULT_SKEW_THRESHOLD
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverJoinTables()
items = spark.table(tables.order_items)
print("Loaded from:", skew_report_module.__file__)
print("Order items:", items.count())

In [ ]:
seller_skew_df = analyze_key_skew(items, "seller_id", DEFAULT_SKEW_THRESHOLD)
product_skew_df = analyze_key_skew(items, "product_id", DEFAULT_SKEW_THRESHOLD)

display(seller_skew_df.limit(10))
display(product_skew_df.limit(10))

In [ ]:
import json

report = run_skew_distribution_report(spark, tables)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/joins_skew_distribution.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)